# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² dataset package using the `mlcroissant` library. All dataset elements are referenced by their Croissant `@id` for consistency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the metadata and initialize the Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and use their `@id` for processing. For each record set, list its fields (columns) and their `@id`s.

In [ ]:
# Retrieve all record sets
record_sets = list(dataset.record_sets())
print(f"Total Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    columns = list(rs.columns)
    print(f"  Number of Columns: {len(columns)}")
    print("  Columns (by @id):")
    for col in columns:
        print(f"    - {col.id} ({col.name})")
    print("-")

## 3. Data Extraction

Load data from a specific record set into a Pandas DataFrame for analysis. All references to record sets and columns use their `@id`. Please adjust `target_record_set_id` below to any available record set (from previous cell).

_Example shown uses the first record set (if available)._


In [ ]:
# Prepare DataFrame extraction for all record sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:
    # Extract all records for this record set
    print(f"Extracting records for record set: {record_set_id}")
    rows = list(dataset.records(record_set=record_set_id))
    if rows:
        df = pd.DataFrame(rows)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows and {len(df.columns)} columns. Cols: {df.columns.tolist()}")
    else:
        print("  No records found or not a tabular set.")

# For demonstration, select the first record set with data (if any)
if dataframes:
    # Select first DataFrame
    demo_record_set_id = next(iter(dataframes.keys()))
    df_demo = dataframes[demo_record_set_id]
    print(f"\nDemo DataFrame columns for record set '{demo_record_set_id}':\n", df_demo.columns.tolist())
    display(df_demo.head())
else:
    print("No record sets with tabular data found.")

## 4. Exploratory Data Analysis (EDA)

We will demonstrate basic EDA operations:
- Filtering records by a numeric field (@id)
- Normalizing a numeric field
- Grouping data by a categorical field (@id)

**Update the `numeric_field_id` and `group_field_id` variables below to use actual column `@id`s present in your chosen record set.**

In [ ]:
# Example EDA on the first data-containing record set, if available

if dataframes:
    df = df_demo.copy()
    # Choose appropriate field @ids below (update if needed)
    # Demo: Select the first numeric-looking column for numeric_field_id
    potential_numeric_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if potential_numeric_ids:
        numeric_field_id = potential_numeric_ids[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalized column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by the first non-numeric field
        group_field_ids = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if group_field_ids:
            group_field_id = group_field_ids[0]
            print(f"\nGrouping by: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Mean of", numeric_field_id, "by", group_field_id)
            display(grouped.head())
        else:
            print("No suitable non-numeric field found to group by.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization

Visualize data distributions or highlight relationships. Adjust the field names (`numeric_field_id`, `group_field_id`) as needed for available columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and potential_numeric_ids:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}' in Record Set '{demo_record_set_id}'")
    plt.show()

    # If group_field_id exists, visualize means by group
    if 'group_field_id' in locals() and group_field_id in df.columns:
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        means.plot(kind='bar', figsize=(10,4))
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and inspect a Croissant-described dataset using the `mlcroissant` library, with all entities referenced by their `@id`. We explored the record sets, extracted tabular data via Croissant's schema, and performed basic EDA with normalization and visualizations. For more complex analysis, adapt field selections and extend this template as the dataset and research question require.